In [3]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import base64
import pandas as pd
import plotly.express as px
from crud import AnimalShelter

# --- Connection ---
# Ensure these credentials match your specific environment
username = "aacuser"
password = "Pa33w0rd"
db = AnimalShelter(username, password)

# --- App Initialization ---
app = JupyterDash(__name__)

# --- Branding ---
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# --- Layout ---
app.layout = html.Div([
    html.Center(html.B(html.H1('Grazioso Salvare Dashboard'))),
    html.Div([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'width': '150px'}),
        html.H4("Lead Developer: Matthew Wood")
    ]),
    html.Hr(),
    
    dcc.Dropdown(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
            {'label': 'Reset', 'value': 'Reset'}
        ],
        value='Reset',
        clearable=False
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        page_action="native",
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single"
    ),
    html.Br(),
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ])
])

# --- Callbacks ---
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'columns'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'Water':
        query = {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}, "sex_upon_outcome": "Intact Female"}
    elif filter_type == 'Mountain':
        query = {"breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}, "sex_upon_outcome": "Intact Male"}
    elif filter_type == 'Disaster':
        query = {"breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}, "sex_upon_outcome": "Intact Male"}
    else:
        query = {}
    
    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns: dff.drop(columns=['_id'], inplace=True)
    
    columns = [{"name": i, "id": i} for i in dff.columns]
    return dff.to_dict('records'), columns

@app.callback(Output('graph-id', "children"), [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None: return []
    dff = pd.DataFrame.from_records(viewData)
    return [dcc.Graph(figure=px.pie(dff, names='breed', title='Preferred Animals'))]

@app.callback(Output('map-id', "children"), [Input('datatable-id', "derived_virtual_data")])
def update_map(viewData):
    if viewData is None: return []
    dff = pd.DataFrame.from_records(viewData)
    # Using MarkerClusterGroup for professional, cleaner map interaction
    return [dl.Map(style={'width': '600px', 'height': '500px'}, center=[30.2672, -97.7431], zoom=10, children=[
        dl.TileLayer(),
        dl.MarkerClusterGroup(children=[
            dl.Marker(position=[row['location_lat'], row['location_long']], children=[
                dl.Tooltip(row['breed']),
                dl.Popup(f"Breed: {row['breed']}")
            ]) for _, row in dff.iterrows()
        ])
    ])]

if __name__ == '__main__':
    # Try port 8000, as it is often permitted in Codio configurations
    app.run_server(host='0.0.0.0', port=8000, debug=False)

 * Running on all addresses.
 * Running on http://10.11.41.77:8000/ (Press CTRL+C to quit)
127.0.0.1 - - [18/Jun/2026 17:39:11] "GET /_alive_d1cb0111-2d9e-4080-8246-453a148e76b1 HTTP/1.1" 200 -


Successfully connected to the MongoDB 'aac' database.
Dash app running on http://0.0.0.0:8000/
